In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [5]:
# Cell 0 — Init + basic fast gates
# Run-time: << 1s for synthetic N~200
import time
t0 = time.time()

import numpy as np
import os, json, math
from numpy.linalg import norm

# === USER CONFIG ===
N = 320            # sample length (adjust; larger -> more precision)
USE_SYNTHETIC = True  # set False to load from file (see below)
INPUT_PATH = "/kaggle/input/mykernel/kernel.csv"  # if USE_SYNTHETIC=False, replace

# === load or synthesize kernel K (real, length N) ===
if USE_SYNTHETIC:
    lam = [0.18, 0.45, 0.72]   # few modes
    w   = [1.0, 0.62, 0.15]
    n = np.arange(N)
    K = np.zeros(N, dtype=float)
    for lj, wj in zip(lam, w):
        K += wj * (1.0 - lj)**n
else:
    # try CSV or npy
    if os.path.exists(INPUT_PATH):
        K = np.loadtxt(INPUT_PATH)[:N]
    else:
        raise FileNotFoundError(f"set USE_SYNTHETIC=True or put kernel at {INPUT_PATH}")

# Ensure shape
K = np.asarray(K, dtype=float)
if K.ndim != 1:
    raise ValueError("Kernel must be 1D real array.")

# === Basic gates (fast) ===
def gate_nonneg(K, tol=1e-12):
    return np.min(K) >= -tol

def gate_fading(K, head=8, tail=24, rel=1e-3):
    if len(K) < head+tail:
        return False, None, None
    h = np.mean(np.abs(K[:head]))
    t = np.mean(np.abs(K[-tail:]))
    return (t <= rel * max(h, 1e-12)) and (np.any(np.abs(K[1:])>1e-12)), float(h), float(t)

def gate_diffusive(K, max_sign_changes=2):
    s = np.sign(K)
    s[np.abs(K) < 1e-12] = 0
    nz = s[s!=0]
    if len(nz) < 4:
        return True, 0
    flips = int(np.sum(nz[:-1]*nz[1:] < 0))
    return flips <= max_sign_changes, flips

ok_nonneg = gate_nonneg(K)
ok_fading, head_mean, tail_mean = gate_fading(K)
ok_diffusive, flips = gate_diffusive(K)

fast_summary = {
    "N": int(N),
    "nonneg": bool(ok_nonneg),
    "fading": bool(ok_fading),
    "diffusive": bool(ok_diffusive),
    "head_mean": head_mean,
    "tail_mean": tail_mean,
    "sign_flips": int(flips)
}

# persist for next cells
np.save("K_cmap.npy", K)
with open("cmap_fast_summary.json","w") as f:
    json.dump(fast_summary, f, indent=2)

print("Cell0 finished — fast checks:", fast_summary)
print("Elapsed %.2fs" % (time.time()-t0))

Cell0 finished — fast checks: {'N': 320, 'nonneg': True, 'fading': True, 'diffusive': True, 'head_mean': 0.7493110386925842, 'tail_mean': 7.074424462329825e-27, 'sign_flips': 0}
Elapsed 0.00s


In [8]:
# === Cell 1 (revised) ===
# Fit finite-order recurrence (H2) via SVD-based order suggestion + ridge LS (robust)
# Run-time: seconds for N~200-500 depending on p_max
import time
t0 = time.time()

import numpy as np
from numpy.linalg import lstsq, svd, norm
import json, math, sys

# -----------------------
# Config (ปรับได้ตามต้องการ)
# -----------------------
p_max = 20             # max order to try
energy_thresh = 1e-4   # SVD relative energy cutoff to suggest model order
ridge_alpha = 1e-8     # ridge regularization
rel_res_threshold = 1e-8  # early stop threshold for relative residual
min_rows_factor = 3    # require at least p * min_rows_factor rows (safety)

# -----------------------
# Load kernel K (เตรียมจาก Cell 0)
# -----------------------
try:
    K = np.load("K_cmap.npy")
except Exception as e:
    raise RuntimeError("ไม่พบไฟล์ K_cmap.npy — กรุณารัน Cell0 ก่อนหรือวางไฟล์ kernel ที่ /mnt/data/K_cmap.npy") from e

K = np.asarray(K, dtype=float)
N = len(K)
if N < 32:
    print("เตือน: N เล็ก ({}). ผลลัพธ์อาจไม่เสถียร".format(N))

# -----------------------
# Build Hankel for model-order suggestion
# -----------------------
# Choose L (rows) as roughly N//2 but at least 8 and not too large
L = max(8, min(N//2, int(N*0.45)))
if L >= N:
    L = max(8, N//2)
cols = N - L + 1
if cols <= 1:
    raise RuntimeError("ไม่สามารถสร้าง Hankel: เพิ่มความยาว N")

# Construct Hankel (L x cols)
H_rows = [K[i:i+cols] for i in range(L)]
H = np.vstack(H_rows)

# SVD (economy)
U, svals, Vh = svd(H, full_matrices=False)
# relative singular values
s_energy = svals / (svals[0] if svals.size>0 else 1.0)
# suggested p by energy threshold (first index where energy drops below threshold)
p_svd = int(np.searchsorted(s_energy, energy_thresh)) + 1
p_svd = max(1, p_svd)
p_guess = min(p_svd, p_max, N//min_rows_factor)

# Safety clamp p_max_actual so rows = N - p >= p*min_rows_factor
p_max_actual = min(p_max, N // (min_rows_factor + 1))
if p_max_actual < 1:
    p_max_actual = 1

# -----------------------
# Robust ridge-fit recurrence function
# -----------------------
def fit_recurrence_ridge(K, p, alpha=1e-8):
    """
    Fit K[n] = sum_{k=1..p} a_k * K[n-k] for n=p..N-1 using ridge-regularized LS.
    Returns (sol, p, rel_res) or (None, None, None) if not enough rows.
    """
    N = len(K)
    rows = N - p
    # require rows sufficiently larger than p to avoid degenerate fit
    if rows <= max(p, 4):
        return None, None, None

    A = np.empty((rows, p), dtype=float)
    b = np.empty(rows, dtype=float)

    # build design matrix explicitly (stable indexing)
    for i in range(rows):
        n = p + i
        # A[i, j] = K[n - (j+1)] for j=0..p-1
        for j in range(p):
            A[i, j] = K[n - (j + 1)]
        b[i] = K[n]

    # ridge solve: (A^T A + alpha I) x = A^T b
    ATA = A.T.dot(A)
    rhs = A.T.dot(b)
    # regularize diagonal adaptively if needed
    ATA_reg = ATA + alpha * np.eye(p)
    try:
        sol = np.linalg.solve(ATA_reg, rhs)
    except np.linalg.LinAlgError:
        sol, *_ = lstsq(ATA_reg, rhs, rcond=None)

    pred = A.dot(sol)
    rel_res = norm(pred - b) / max(1e-12, norm(b))
    return sol, p, float(rel_res)


# -----------------------
# Search best p (1..p_max_actual), prefer small p with low residual
# Try values around p_guess first for speed, then expand
# -----------------------
best = None
tried = set()

# create an ordered list of p candidates: p_guess, then increasing around it
candidates = []
# start with 1..min(6,p_max_actual) for small models
for q in range(1, min(6, p_max_actual)+1):
    candidates.append(q)
# then include p_guess..p_guess+range
for q in range(max(1, p_guess-2), min(p_max_actual, p_guess+4)+1):
    if q not in candidates:
        candidates.append(q)
# finally fill remaining up to p_max_actual
for q in range(1, p_max_actual+1):
    if q not in candidates:
        candidates.append(q)

for p in candidates:
    if p in tried:
        continue
    tried.add(p)
    sol, p_try, rel = fit_recurrence_ridge(K, p, alpha=ridge_alpha)
    if sol is None:
        continue
    if best is None or rel < best[2] or (abs(rel - best[2])<1e-12 and p < best[1]):
        best = (sol, p_try, rel)
    # early stop if residual extremely small
    if rel < rel_res_threshold:
        break

if best is None:
    raise RuntimeError("Recurrence fit failed for all p candidates; increase N or p_max or inspect K")

a_coeffs, p_fitted, rel_res = best
a_coeffs = np.asarray(a_coeffs, dtype=float)

# persistence: save results for downstream cells
rec_result = {"p_fitted": int(p_fitted), "rel_res": float(rel_res), "a_coeffs": [float(x) for x in a_coeffs],
              "p_svd": int(p_svd), "p_max_actual": int(p_max_actual), "L_hankel": int(L)}
with open("cmap_recurrence.json","w") as f:
    json.dump(rec_result, f, indent=2)

# print summary
print("Cell1 finished: N={}, L={}, p_svd_suggest={}, p_fitted={}, rel_res={:.3e}".format(
    N, L, p_svd, p_fitted, rel_res))
print("a_coeffs (first 10):", np.round(a_coeffs[:min(10,len(a_coeffs))], 6))
print("Elapsed %.2fs" % (time.time() - t0))

Cell1 finished: N=320, L=144, p_svd_suggest=145, p_fitted=20, rel_res=1.218e-06
a_coeffs (first 10): [0.0012   0.001463 0.001782 0.00217  0.00264  0.003209 0.003893 0.004711
 0.005678 0.006804]
Elapsed 0.06s


In [11]:
# === Cell 2 (FINAL) ===
# Single-cell, "accepted-level" numerical decomposition + model reduction + bootstrap stability
# - Loads K_cmap.npy (created in Cell0)
# - Loads cmap_recurrence.json if present (created in Cell1), else performs a compact fallback fit
# - Extracts roots, solves residues, computes reconstruction
# - Identifies significant modes, refits reduced model, runs bootstrap on reduced residues
# - Produces JSON outputs: cmap_decomp_final.json and cmap_sig_modes.json
#
# Runtime: tuned to be fast for N~200-1000 and p_reduced small (<=10). Uses numpy + scipy.
# Usage: paste as a single notebook cell and run.

import time
t_start = time.time()

import os, json, math
import numpy as np
from numpy.linalg import lstsq, norm
from numpy.linalg import eigvals
from numpy import vander
from numpy.random import default_rng

rng = default_rng(123456)

# ---------------------------
# Config (tune for runtime/robustness)
# ---------------------------
N_use = None           # if int, will truncate/extend K to this length; if None uses full length
p_max_fallback = 20    # only used if cmap_recurrence.json missing
n_boot = 100           # bootstrap iterations for reduced model (100 is modest; increase if budget)
noise_frac_list = [1e-4, 1e-3, 1e-2]  # noise levels to test sensitivity
rel_thresh_mode = 0.01 # relative threshold (1% of max residue) for selecting significant modes
abs_floor = 1e-6       # absolute floor for residue significance
recon_tol_refit = 1e-6 # desired reconstruction tolerance for reduced refit

# ---------------------------
# Load kernel K
# ---------------------------
if not os.path.exists("K_cmap.npy"):
    raise FileNotFoundError("K_cmap.npy not found — run Cell0 first or place kernel at ./K_cmap.npy")
K = np.load("K_cmap.npy").astype(float)
if N_use is not None:
    K = K[:N_use]
N = len(K)
if N < 16:
    raise ValueError("Kernel too short (N < 16); increase N for reliable decomposition")

# ---------------------------
# Load recurrence if present, otherwise fallback fit of modest p
# ---------------------------
a_coeffs = None
p_fitted = None
if os.path.exists("cmap_recurrence.json"):
    with open("cmap_recurrence.json") as f:
        rec = json.load(f)
    a_coeffs = np.array(rec.get("a_coeffs", []), dtype=float)
    p_fitted = int(rec.get("p_fitted", len(a_coeffs))) if a_coeffs.size>0 else None

if a_coeffs is None or a_coeffs.size == 0:
    # fallback: fit small-order recurrence via ridge LS (robust)
    def fit_recurrence_ridge_simple(K, p, alpha=1e-8):
        N = len(K)
        rows = N - p
        if rows <= max(p, 4):
            return None, None
        A = np.empty((rows, p), dtype=float)
        b = np.empty(rows, dtype=float)
        for i in range(rows):
            n = p + i
            for j in range(p):
                A[i, j] = K[n - (j + 1)]
            b[i] = K[n]
        ATA = A.T.dot(A)
        rhs = A.T.dot(b)
        ATA_reg = ATA + alpha * np.eye(p)
        try:
            sol = np.linalg.solve(ATA_reg, rhs)
        except Exception:
            sol, *_ = lstsq(ATA_reg, rhs, rcond=None)
        pred = A.dot(sol)
        rel = float(norm(pred - b) / max(1e-12, norm(b)))
        return sol, rel

    best = None
    for p_try in range(1, min(p_max_fallback, N//4)+1):
        sol, rel = fit_recurrence_ridge_simple(K, p_try, alpha=1e-8)
        if sol is None:
            continue
        if best is None or rel < best[2]:
            best = (sol, p_try, rel)
        if rel < 1e-6:
            break
    if best is None:
        raise RuntimeError("Fallback recurrence fit failed; increase N or provide cmap_recurrence.json")
    a_coeffs, p_fitted, rel_fit = best[0], best[1], best[2]
    a_coeffs = np.asarray(a_coeffs, dtype=float)
    # persist fallback result for reproducibility
    with open("cmap_recurrence.json","w") as f:
        json.dump({"p_fitted": int(p_fitted), "rel_res": float(rel_fit), "a_coeffs": [float(x) for x in a_coeffs]}, f, indent=2)

# ---------------------------
# Build polynomial A(z) = z^p - a1 z^{p-1} - ... - ap and extract roots
# ---------------------------
p = int(p_fitted)
A_poly = np.zeros(p+1, dtype=complex)
A_poly[0] = 1.0
for i, ai in enumerate(a_coeffs, start=1):
    A_poly[i] = -ai
roots = np.roots(A_poly)  # z-domain roots
J = len(roots)

# ---------------------------
# Solve residues via Vandermonde on first J samples (or a slightly larger Jv for stability)
# ---------------------------
Jv = min(J, max(3, N//4))
V = np.vander(roots, N=Jv, increasing=True).T[:Jv, :Jv]
# if V is singular or poorly conditioned, use least squares on first 2*J samples
use_ls = False
condV = np.linalg.cond(V) if V.size>0 else 1.0
if condV > 1e12:
    use_ls = True
if use_ls:
    # build tall Vandermonde for n=0..M-1 where M = min(N, 3*J)
    M = min(N, max(2*J, 3*J))
    Vtall = np.vander(roots, N=M, increasing=True).T[:M, :J]
    cb, *_ = lstsq(Vtall, K[:M], rcond=None)
else:
    try:
        cb = np.linalg.solve(V, K[:Jv])
    except Exception:
        cb, *_ = lstsq(V, K[:Jv], rcond=None)

# full reconstruction and rel error
n = np.arange(N)
K_recon = np.zeros(N, dtype=complex)
for cj, rj in zip(cb, roots):
    K_recon += cj * (rj**n)
recon_rel_err = float(norm(K_recon.real - K) / max(1e-12, norm(K)))

# assemble pole/residue list (serializable)
poles_info = []
for idx, (rj, cj) in enumerate(zip(roots, cb)):
    poles_info.append({
        "index": int(idx),
        "z": [float(np.real(rj)), float(np.imag(rj))],
        "abs_z": float(abs(rj)),
        "lambda": [float(1.0 - np.real(rj)), float(-np.imag(rj))],
        "residue": [float(np.real(cj)), float(np.imag(cj))],
        "is_real": bool(abs(np.imag(rj)) < 1e-10),
        "condV": float(condV) if idx==0 else None
    })

# ---------------------------
# Bootstrap residues stability on full set of roots (fast, small noise) — use small n_boot to assess global noise sensitivity
# ---------------------------
boot_res_means = None
boot_res_stds = None
if n_boot > 0:
    B = n_boot
    boot_out = np.zeros((B, len(cb)), dtype=float)
    for bi in range(B):
        # draw noise relative to mean(abs(K))
        noise_level = 1e-3  # small for initial pass; we'll test multiple levels later
        noise = noise_level * (np.mean(np.abs(K)) + 1e-12) * rng.standard_normal(size=K.shape)
        Kb = K + noise
        if use_ls:
            try:
                cb_b, *_ = lstsq(Vtall, Kb[:Vtall.shape[0]], rcond=None)
            except Exception:
                cb_b = np.zeros_like(cb)
        else:
            try:
                cb_b, *_ = lstsq(V, Kb[:Jv], rcond=None)
            except Exception:
                cb_b = np.zeros_like(cb)
        boot_out[bi, :len(cb_b)] = np.real(cb_b)
    boot_res_means = list(np.mean(boot_out, axis=0))
    boot_res_stds  = list(np.std(boot_out, axis=0))
else:
    boot_res_means = [float(np.real(x)) for x in cb]
    boot_res_stds = [0.0]*len(cb)

# ---------------------------
# Identify significant modes (relative + absolute threshold)
# ---------------------------
res_real = np.array([float(np.real(x)) for x in cb])
abs_res = np.abs(res_real)
max_res = abs_res.max() if abs_res.size>0 else 0.0
threshold = max(rel_thresh_mode * max_res, abs_floor)
significant_idx = np.where(abs_res >= threshold)[0]
# if none found (unlikely), fall back to top-3 by magnitude
if significant_idx.size == 0:
    significant_idx = np.argsort(-abs_res)[:min(3, len(abs_res))]

significant_idx = list(map(int, significant_idx))
# sort by descending residue magnitude
significant_idx = sorted(significant_idx, key=lambda j: -abs_res[j])

# ---------------------------
# Refit reduced model using only significant roots (refined residues via least squares)
# ---------------------------
sig_roots = roots[significant_idx]
Jsig = len(sig_roots)
if Jsig == 0:
    raise RuntimeError("No significant modes detected — cannot form reduced model")

# Build Vandermonde (tall) using first M samples for LS (M = min(N, max(5*Jsig, 3*Jsig)))
M = min(N, max(5*Jsig, 3*Jsig))
Vred = np.vander(sig_roots, N=M, increasing=True).T[:M, :Jsig]
try:
    cres, *_ = lstsq(Vred, K[:M], rcond=None)
except Exception:
    cres = np.zeros(Jsig, dtype=complex)

# full reconstruction from reduced model and relative error
K_recon_red = np.zeros(N, dtype=complex)
for cj, rj in zip(cres, sig_roots):
    K_recon_red += cj * (rj**n)
recon_red_rel_err = float(norm(K_recon_red.real - K) / max(1e-12, norm(K)))

# If reduced reconstruction error is poor, try small regularization and refit
if recon_red_rel_err > recon_tol_refit and Jsig <= 8:
    # ridge solve on Vred^T Vred
    A = Vred
    ATA = A.T.dot(A)
    rhs = A.T.dot(K[:M])
    alpha_ridge = 1e-8
    try:
        cres = np.linalg.solve(ATA + alpha_ridge*np.eye(Jsig), rhs)
    except Exception:
        cres, *_ = lstsq(ATA + alpha_ridge*np.eye(Jsig), rhs, rcond=None)
    K_recon_red = np.zeros(N, dtype=complex)
    for cj, rj in zip(cres, sig_roots):
        K_recon_red += cj * (rj**n)
    recon_red_rel_err = float(norm(K_recon_red.real - K) / max(1e-12, norm(K)))

# ---------------------------
# Bootstrap for reduced model across several noise levels (robustness check)
# ---------------------------
reduced_boot_stats = {}
for noise_frac in noise_frac_list:
    B = min(200, n_boot) if n_boot>0 else 0
    if B <= 0:
        reduced_boot_stats[float(noise_frac)] = {"mean": [float(np.real(x)) for x in cres],
                                                 "std": [0.0]*len(cres)}
        continue
    boot_c = np.zeros((B, Jsig), dtype=float)
    for bi in range(B):
        noise = noise_frac * (np.mean(np.abs(K)) + 1e-12) * rng.standard_normal(size=K.shape)
        Kb = K + noise
        # solve least squares for reduced roots
        try:
            cb_b, *_ = lstsq(Vred, Kb[:M], rcond=None)
        except Exception:
            cb_b = np.zeros(Jsig, dtype=complex)
        boot_c[bi, :] = np.real(cb_b)
    reduced_boot_stats[float(noise_frac)] = {"mean": [float(x) for x in np.mean(boot_c, axis=0)],
                                             "std":  [float(x) for x in np.std(boot_c, axis=0)],
                                             "n": int(B)}

# ---------------------------
# Prepare final JSON-friendly reports
# ---------------------------
sig_modes = []
for idx_local, j in enumerate(significant_idx):
    rj = roots[j]
    cj_full = cb[j] if j < len(cb) else 0.0
    # find corresponding index in reduced model (match by nearest root)
    # compute index in reduced cres by minimal distance
    dists = np.abs(np.array(list(sig_roots)) - rj)
    red_pos = int(np.argmin(dists))
    cres_val = cres[red_pos] if red_pos < len(cres) else 0.0
    sig_modes.append({
        "orig_index": int(j),
        "reduced_index": int(red_pos),
        "z": [float(np.real(rj)), float(np.imag(rj))],
        "lambda": float(1.0 - float(np.real(rj))),
        "residue_full": float(np.real(cj_full)),
        "residue_reduced": float(np.real(cres_val)),
        "bootstrap_mean_reduced": {str(k): v["mean"][red_pos] for k,v in reduced_boot_stats.items()},
        "bootstrap_std_reduced":  {str(k): v["std"][red_pos]  for k,v in reduced_boot_stats.items()}
    })

final_report = {
    "meta": {
        "elapsed_s": float(time.time() - t_start),
        "N": int(N),
        "p_original": int(p),
        "J_original": int(J),
        "J_reduced": int(Jsig)
    },
    "reconstruction": {
        "full_rel_error": float(recon_rel_err),
        "reduced_rel_error": float(recon_red_rel_err)
    },
    "poles_info_full": poles_info,
    "significant_indices": [int(x) for x in significant_idx],
    "reduced_roots": [[float(np.real(z)), float(np.imag(z))] for z in sig_roots],
    "reduced_residues": [float(np.real(x)) for x in cres],
    "reduced_bootstrap": reduced_boot_stats,
    "sig_modes_summary": sig_modes
}

# Save outputs
with open("cmap_decomp_final.json", "w") as f:
    json.dump(final_report, f, indent=2)

with open("cmap_sig_modes.json", "w") as f:
    json.dump({"p_reduced": int(Jsig), "sig_modes": sig_modes}, f, indent=2)

# Human-readable summary
print("=== CMAP: FINAL DECOMPOSITION (cell) ===")
print(f"Elapsed: {final_report['meta']['elapsed_s']:.2f}s | N={N} | p_orig={p} | J_orig={J} | J_reduced={Jsig}")
print(f"Full reconstruction rel error: {final_report['reconstruction']['full_rel_error']:.3e}")
print(f"Reduced reconstruction rel error: {final_report['reconstruction']['reduced_rel_error']:.3e}")
print("Significant mode indices (original):", final_report["significant_indices"])
for m in final_report["sig_modes_summary"]:
    print(f" - mode orig_idx={m['orig_index']} -> lambda={m['lambda']:.6f}, residue_reduced={m['residue_reduced']:.6g}, bootstrap_std_examples={ {k:round(v,3) for k,v in m['bootstrap_std_reduced'].items()} }")

print("\nSaved: cmap_decomp_final.json and cmap_sig_modes.json")
print("You can now run the Toeplitz passivity tests (Cell 3) and produce T0 constructive bounds for final certification.")

=== CMAP: FINAL DECOMPOSITION (cell) ===
Elapsed: 0.06s | N=320 | p_orig=20 | J_orig=20 | J_reduced=3
Full reconstruction rel error: 1.581e-08
Reduced reconstruction rel error: 8.817e-07
Significant mode indices (original): [17, 18, 19]
 - mode orig_idx=17 -> lambda=0.180000, residue_reduced=1, bootstrap_std_examples={'0.0001': 0.0, '0.001': 0.0, '0.01': 0.0}
 - mode orig_idx=18 -> lambda=0.450023, residue_reduced=0.620119, bootstrap_std_examples={'0.0001': 0.0, '0.001': 0.0, '0.01': 0.001}
 - mode orig_idx=19 -> lambda=0.720142, residue_reduced=0.149878, bootstrap_std_examples={'0.0001': 0.0, '0.001': 0.0, '0.01': 0.001}

Saved: cmap_decomp_final.json and cmap_sig_modes.json
You can now run the Toeplitz passivity tests (Cell 3) and produce T0 constructive bounds for final certification.


In [12]:
# === Cell 3 (FINAL) ===
# Toeplitz PSD passivity checks + constructive T0 bounds + final verdict
# Inputs (must exist): K_cmap.npy, cmap_decomp_final.json, cmap_sig_modes.json, cmap_recurrence.json
# Outputs: cmap_full_report.json
import time, json, math, os
import numpy as np
from numpy.linalg import eigvalsh, norm
t0 = time.time()

# Parameters / tolerances
toeplitz_horizons = [40, 80, 160]   # horizons to test (will clamp to N)
toeplitz_tol = -1e-8                # allowed tiny negative eigen due to numerics
recon_tol_reduced = 5e-4            # acceptable rel error for reduced reconstruction
residue_neg_tol = -1e-8             # allowed tiny negative numeric residue
T0_sanity_cap = 100000              # clamp huge T0
report = {}

# ---- load inputs ----
if not os.path.exists("K_cmap.npy"):
    raise FileNotFoundError("K_cmap.npy missing (run Cell0)")
K = np.load("K_cmap.npy").astype(float)
N = len(K)

if not os.path.exists("cmap_decomp_final.json"):
    raise FileNotFoundError("cmap_decomp_final.json missing (run Cell2 FINAL)")
with open("cmap_decomp_final.json") as f:
    decomp = json.load(f)

if not os.path.exists("cmap_sig_modes.json"):
    raise FileNotFoundError("cmap_sig_modes.json missing (run Cell2 FINAL)")
with open("cmap_sig_modes.json") as f:
    sigm = json.load(f)

# try load recurrence (optional for metadata)
rec = {}
if os.path.exists("cmap_recurrence.json"):
    with open("cmap_recurrence.json") as f:
        rec = json.load(f)

# ---- unpack decomp ----
full_rel_err = float(decomp["reconstruction"]["full_rel_error"])
reduced_rel_err = float(decomp["reconstruction"]["reduced_rel_error"])
p_orig = int(decomp["meta"]["p_original"])
J_orig = int(decomp["meta"]["J_original"])
J_reduced = int(decomp["meta"]["J_reduced"])
poles_full = decomp["poles_info_full"]
reduced_roots = np.array(decomp["reduced_roots"], dtype=float)  # shape (Jr,2)
reduced_residues = np.array(decomp["reduced_residues"], dtype=float)  # Jr
sig_indices = decomp["significant_indices"]

# reconstruct reduced-model kernel (for Toeplitz test)
n = np.arange(N)
K_recon_red = np.zeros(N, dtype=float)
for (zr_real, zr_im), cj in zip(reduced_roots, reduced_residues):
    rj = complex(zr_real, zr_im)
    # use real part of reconstruction (modes expected real or near-real)
    K_recon_red += np.real(cj * (rj**n))

# ---- Toeplitz causal matrix builder ----
def toeplitz_causal(Kvec, M):
    # build MxM lower-triangular Toeplitz T where T[n,m] = K[n-m] for n>=m
    T = np.zeros((M, M), dtype=float)
    for n0 in range(M):
        for m0 in range(n0+1):
            idx = n0 - m0
            if idx < len(Kvec):
                T[n0, m0] = float(Kvec[idx])
            else:
                T[n0, m0] = 0.0
    # symmetrize to avoid numerical asymmetry
    return 0.5 * (T + T.T)

toeplitz_results = {"data": {}, "recon_reduced": {}}
toeplitz_ok_data = True
toeplitz_ok_recon = True

for M in toeplitz_horizons:
    Mclamp = min(M, N)
    if Mclamp < 2:
        continue
    # data Toeplitz
    S = toeplitz_causal(K, Mclamp)
    ev = eigvalsh(S)
    min_eig = float(np.min(ev))
    num_neg = int(np.sum(ev < -1e-12))
    toeplitz_results["data"][f"M={Mclamp}"] = {"min_eig": min_eig, "num_neg": num_neg}
    if min_eig < toeplitz_tol:
        toeplitz_ok_data = False
    # reduced reconstruction Toeplitz
    S2 = toeplitz_causal(K_recon_red, Mclamp)
    ev2 = eigvalsh(S2)
    min_eig2 = float(np.min(ev2))
    num_neg2 = int(np.sum(ev2 < -1e-12))
    toeplitz_results["recon_reduced"][f"M={Mclamp}"] = {"min_eig": min_eig2, "num_neg": num_neg2}
    if min_eig2 < toeplitz_tol:
        toeplitz_ok_recon = False

# ---- Constructive T0 bounds (use reduced model: roots = r_j, residues = c_j) ----
# formula: T0_j = ceil(e^2 * 2 * Cj / (|cj| * (1 - rj)^2)) + 1, where
# Cj = sum_{l != j} |c_l| / (1 - rho_jl)^2, rho_jl = max(rj^2, rj*rl)
T0_list = []
rj_vals = np.array([z[0] + 1j*z[1] for z in reduced_roots], dtype=complex)
cj_vals = np.array(reduced_residues, dtype=float)

for j in range(len(rj_vals)):
    rj = rj_vals[j]
    cj = cj_vals[j]
    # skip tiny cj
    if abs(cj) < 1e-14:
        T0_list.append({"index": int(j), "note": "cj ~ 0", "T0": None})
        continue
    Cj = 0.0
    for l in range(len(rj_vals)):
        if l == j:
            continue
        rl = rj_vals[l]
        rho = max(abs(rj)**2, abs(rj*rl))
        denom = (1.0 - rho)
        if denom <= 1e-12:
            # extremely close roots => large Cj, clamp behavior
            Cj += abs(float(cj_vals[l])) / (1e-12**2)
        else:
            Cj += abs(float(cj_vals[l])) / (denom**2)
    denom_main = abs(cj) * (abs(1.0 - rj))**2
    if denom_main <= 1e-16:
        T0 = T0_sanity_cap
    else:
        T0 = math.ceil((math.e**2 * 2.0 * Cj) / denom_main) + 1
        if T0 > T0_sanity_cap:
            T0 = T0_sanity_cap
    T0_list.append({"index": int(j), "r_j": float(np.real(rj)), "lambda_j": float(np.real(1.0 - rj)),
                    "c_j": float(cj), "Cj": float(Cj), "T0": int(T0)})

# ---- Final verdict logic ----
verdict_reasons = []
# basic numeric gates
if not (full_rel_err >= 0):
    verdict_reasons.append("Invalid full reconstruction error")
# reduced reconstruction must be small
if reduced_rel_err > recon_tol_reduced:
    verdict_reasons.append(f"Reduced reconstruction error too large: {reduced_rel_err:.3e} > {recon_tol_reduced:.1e}")
# residues non-negative (allow tiny negative numeric noise)
neg_residues = [float(x) for x in reduced_residues if float(x) < residue_neg_tol]
if len(neg_residues) > 0:
    verdict_reasons.append(f"Negative residues found (beyond tol): {neg_residues}")
# lambdas in (0,1) for each reduced root (real parts)
for j, r in enumerate(rj_vals):
    if not (0.0 < np.real(r) < 1.0):
        verdict_reasons.append(f"Reduced root r_j not in (0,1) (j={j}, r={np.real(r):.6g})")
# toeplitz passivity both on data and reduced recon
if not toeplitz_ok_data:
    verdict_reasons.append("Toeplitz passivity failed on original data (negative eigenvalues at some horizon)")
if not toeplitz_ok_recon:
    verdict_reasons.append("Toeplitz passivity failed on reduced reconstruction (negative eigenvalues)")

final_pass = (len(verdict_reasons) == 0)

final_msg = "PASS: CMAP numeric certificate candidate" if final_pass else "FAIL: CMAP numeric checks not fully satisfied"

# ---- Assemble full report and save ----
full_report = {
    "meta": {"elapsed_cell_s": float(time.time() - t0), "N": int(N), "p_orig": int(p_orig), "J_reduced": int(J_reduced)},
    "errors": {"full_rel_err": float(full_rel_err), "reduced_rel_err": float(reduced_rel_err)},
    "toeplitz": toeplitz_results,
    "T0_list": T0_list,
    "verdict": {"pass": bool(final_pass), "message": final_msg, "reasons": verdict_reasons}
}

with open("cmap_full_report.json", "w") as f:
    json.dump(full_report, f, indent=2)

# ---- human-readable summary ----
print("=== CMAP: FINAL CERTIFICATION ===")
print(f"Elapsed cell time: {full_report['meta']['elapsed_cell_s']:.2f}s")
print(f"N={N}, p_orig={p_orig}, J_reduced={J_reduced}")
print(f"Full reconstruction rel error: {full_rel_err:.3e}")
print(f"Reduced reconstruction rel error: {reduced_rel_err:.3e}")
print("Toeplitz (data) min eigenvalues:")
for k,v in full_report['toeplitz']['data'].items():
    print(f" - {k}: min_eig={v['min_eig']:.3e}, num_neg={v['num_neg']}")
print("Toeplitz (reduced recon) min eigenvalues:")
for k,v in full_report['toeplitz']['recon_reduced'].items():
    print(f" - {k}: min_eig={v['min_eig']:.3e}, num_neg={v['num_neg']}")

print("\nConstructive T0 bounds (reduced modes):")
for t in full_report['T0_list']:
    print(f" - mode {t['index']}: lambda={t['lambda_j']:.6f}, c_j={t['c_j']:.6g}, T0={t['T0']}, Cj={t['Cj']:.3e}")

print("\nFINAL VERDICT:", full_report['verdict']['message'])
if not final_pass:
    print("Reasons:")
    for r in verdict_reasons:
        print(" -", r)
else:
    print("All strict numeric checks passed for CMAP candidate. You can attach cmap_full_report.json and cmap_decomp_final.json as reproducible artifacts for reviewers.")

print("\nSaved ./cmap_full_report.json")

=== CMAP: FINAL CERTIFICATION ===
Elapsed cell time: 0.04s
N=320, p_orig=20, J_reduced=3
Full reconstruction rel error: 1.581e-08
Reduced reconstruction rel error: 8.817e-07
Toeplitz (data) min eigenvalues:
 - M=40: min_eig=1.067e+00, num_neg=0
 - M=80: min_eig=1.067e+00, num_neg=0
 - M=160: min_eig=1.067e+00, num_neg=0
Toeplitz (reduced recon) min eigenvalues:
 - M=40: min_eig=1.067e+00, num_neg=0
 - M=80: min_eig=1.067e+00, num_neg=0
 - M=160: min_eig=1.067e+00, num_neg=0

Constructive T0 bounds (reduced modes):
 - mode 0: lambda=0.180000, c_j=1, T0=3274, Cj=7.175e+00
 - mode 1: lambda=0.450023, c_j=0.620119, T0=428, Cj=3.626e+00
 - mode 2: lambda=0.720142, c_j=0.149878, T0=486, Cj=2.551e+00

FINAL VERDICT: PASS: CMAP numeric certificate candidate
All strict numeric checks passed for CMAP candidate. You can attach cmap_full_report.json and cmap_decomp_final.json as reproducible artifacts for reviewers.

Saved ./cmap_full_report.json
